# 01 — Classical GAN Baseline (SMI log-returns)

**ZHAW CEC Quantum Computing — Final project**

This notebook trains a small classical GAN on Swiss Market Index log-return windows and saves results to `results/classical/` for later head-to-head comparison with the hybrid quantum GAN.

All heavy lifting lives in `src/` so this notebook stays short.

## Setup (Colab)

Skip cells with `# colab-only` if running locally.

In [ ]:
# colab-only: clone repo & install deps. Comment out when running locally.
# !git clone https://github.com/wuns/qGAN-market-generator.git
# %cd qGAN-market-generator
# !pip install -q -r requirements.txt

In [ ]:
import sys, pathlib, json
# Make src/ importable whether running locally from notebooks/ or from Colab root.
ROOT = pathlib.Path.cwd()
if (ROOT / 'src').is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'src').is_dir():
    sys.path.insert(0, str(ROOT.parent))
    ROOT = ROOT.parent

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data       import prepare_smi_data
from src.models     import ClassicalGenerator, Discriminator, count_parameters
from src.training   import set_seed, make_dataloader, train_gan, generate
from src.evaluation import (summarise, ks_distance, plot_distributions,
                            plot_acf_comparison, plot_sample_paths, build_report)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device, '| repo root =', ROOT)

## Config

In [ ]:
SEED       = 42
WINDOW     = 20
LATENT_DIM = 8
EPOCHS     = 80
BATCH      = 64

set_seed(SEED)

## Data

In [ ]:
data = prepare_smi_data(
    window=WINDOW,
    cache_path=ROOT / 'results' / 'smi_prices.pkl',  # speeds up repeat runs
)
print(f'Train windows: {data.train_windows.shape}')
print(f'Test  windows: {data.test_windows.shape}')
print(f'Scale (4 sigma): {data.scale:.5f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(data.raw_returns, lw=0.4); ax[0].set_title('SMI log-returns'); ax[0].set_ylabel('r_t')
ax[1].hist(data.raw_returns, bins=80); ax[1].set_title('Distribution'); ax[1].set_xlabel('r_t')
plt.tight_layout(); plt.show()

## Model & training

In [ ]:
G = ClassicalGenerator(latent_dim=LATENT_DIM, window=WINDOW)
D = Discriminator(window=WINDOW)
n_params_G = count_parameters(G)
n_params_D = count_parameters(D)
print(f'Generator params:     {n_params_G}')
print(f'Discriminator params: {n_params_D}')

loader = make_dataloader(data.train_windows, batch_size=BATCH)
history = train_gan(G, D, loader, latent_dim=LATENT_DIM, epochs=EPOCHS, device=device)
print(f'\nTraining time: {history.train_time_sec:.1f} s')

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(history.d_loss, label='D loss')
plt.plot(history.g_loss, label='G loss')
plt.xlabel('epoch'); plt.legend(); plt.title('GAN training losses')
plt.tight_layout(); plt.show()

## Generate & evaluate

In [ ]:
n_eval = len(data.test_windows)
fake_scaled, t_inf = generate(G, n_eval, LATENT_DIM, device=device)
fake_returns = data.unscale(fake_scaled)
real_returns = data.unscale(data.test_windows)
samples_per_sec = (n_eval * WINDOW) / t_inf
print(f'Inference: {n_eval} windows in {t_inf*1000:.1f} ms ({samples_per_sec:.0f} samples/s)')

In [ ]:
plot_sample_paths(real_returns, fake_returns); plt.tight_layout(); plt.show()
plot_distributions(real_returns, fake_returns); plt.tight_layout(); plt.show()
plot_acf_comparison(real_returns, fake_returns); plt.tight_layout(); plt.show()

In [ ]:
report = build_report(
    real_returns=real_returns,
    fake_returns=fake_returns,
    n_params_G=n_params_G,
    n_params_D=n_params_D,
    train_time_sec=history.train_time_sec,
    inference_samples_per_sec=samples_per_sec,
    extras={'seed': SEED, 'epochs': EPOCHS, 'window': WINDOW, 'latent_dim': LATENT_DIM,
            'model': 'classical_mlp_gan'},
)
for k, v in report.items():
    print(f'{k:30s} {v}')

## Save artefacts

These will be loaded by `03_comparison.ipynb`.

In [ ]:
out = ROOT / 'results' / 'classical'; out.mkdir(parents=True, exist_ok=True)
torch.save(G.state_dict(), out / 'generator.pt')
np.save(out / 'fake_returns.npy', fake_returns)
np.save(out / 'real_returns_test.npy', real_returns)
np.save(out / 'scale.npy', np.array([data.scale]))
with open(out / 'metrics.json', 'w') as f:
    json.dump(report, f, indent=2, default=float)
print('Saved to', out)

## Next

Notebook `02_quantum_gan.ipynb` will reuse the same `Discriminator`, `train_gan`, and evaluation functions, replacing only the generator with a small variational quantum circuit.